## Examen Segundo Bimestre RI
### **Estudiante:** Kevin Alvear

#### Instalacion de dependencias

In [ ]:
# Instalacion de librerias necesarias
# pip install sentence-transformers faiss-cpu nltk google-generativeai streamlit pandas numpy

#### Importacion de librerias

In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sentence_transformers import SentenceTransformer
import faiss
import os
import google.generativeai as genai
import json
import time

# Descargar recursos de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Kevin
[nltk_data]     Alvear\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

#### Carga del dataset

In [3]:
# El dataset se encuentra en la ruta de Kaggle
file_path = 'data/arxiv_data.csv'
df = pd.read_csv(file_path, nrows=25000)

print("Dataset cargado correctamente")
print(f"Total de registros: {len(df)}")
print(f"Columnas disponibles: {df.columns.tolist()}")
print("Muestra de los datos:")
df.head()

Dataset cargado correctamente
Total de registros: 25000
Columnas disponibles: ['titles', 'summaries', 'terms']
Muestra de los datos:


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


#### Exploracion inicial del corpus

In [4]:
# Estadisticas basicas del corpus
print("Estadisticas del corpus:")
print(f"- Documentos totales: {len(df)}")
print(f"- Terminos unicos: {df['terms'].nunique()}")
print(f"- Promedio de palabras en summaries: {df['summaries'].str.split().str.len().mean():.0f}")
print(f"- Ejemplo de terminos: {df['terms'].iloc[0]}")

# Verificar datos faltantes
print(f"- Summaries faltantes: {df['summaries'].isna().sum()}")
print(f"- Titles faltantes: {df['titles'].isna().sum()}")
print(f"- Terms faltantes: {df['terms'].isna().sum()}")

Estadisticas del corpus:
- Documentos totales: 25000
- Terminos unicos: 1801
- Promedio de palabras en summaries: 174
- Ejemplo de terminos: ['cs.CV', 'cs.LG']
- Summaries faltantes: 0
- Titles faltantes: 0
- Terms faltantes: 0


#### Funcion de limpieza de texto

In [5]:
def clean_text(text):
    """
    Limpia y normaliza el texto para generar embeddings
    """
    if pd.isna(text):
        return ""

    # Convertir a minusculas y eliminar caracteres especiales
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())

    # Tokenizacion
    tokens = nltk.word_tokenize(text)

    # Eliminar stopwords y aplicar lematizacion
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    cleaned_tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]

    return " ".join(cleaned_tokens)

#### Preprocesamiento del corpus

In [6]:
# Aplicar limpieza a titles y summaries para tener mas contexto
print("Iniciando preprocesamiento de textos...")
df['clean_title'] = df['titles'].apply(clean_text)
df['clean_summary'] = df['summaries'].apply(clean_text)

# Combinar title y summary para crear un texto mas rico para embeddings
df['combined_text'] = df['clean_title'] + " " + df['clean_summary']

print("Preprocesamiento completado")

# Verificar resultado
print("Ejemplo de texto combinado:")
print(f"Titulo original: {df['titles'].iloc[0][:80]}...")
print(f"Summary original: {df['summaries'].iloc[0][:100]}...")
print(f"Texto combinado: {df['combined_text'].iloc[0][:200]}...")

Iniciando preprocesamiento de textos...
Preprocesamiento completado
Ejemplo de texto combinado:
Titulo original: Survey on Semantic Stereo Matching / Semantic Depth Estimation...
Summary original: Stereo matching is one of the widely used techniques for inferring depth from
stereo images owing to...
Texto combinado: survey semantic stereo matching semantic depth estimation stereo matching one widely used technique inferring depth stereo image owing robustness speed become one major topic research since find appli...


#### Carga del modelo de embeddings

In [8]:
# Cargar modelo preentrenado de Sentence Transformers
print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Modelo cargado correctamente")

# Verificar dimension del modelo
print(f"Dimension del embedding: {embedding_model.get_sentence_embedding_dimension()}")

Cargando modelo de embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6783.70it/s]


Modelo cargado correctamente
Dimension del embedding: 384


C:\Users\Kevin Alvear\AppData\Local\Temp\ipykernel_10208\1920355367.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Dimension del embedding: {embedding_model.get_sentence_embedding_dimension()}")


#### Generacion de embeddings

In [9]:
# Generar embeddings usando el texto combinado (title + summary)
print("Generando embeddings para el corpus...")
print("Este proceso puede tomar algunos minutos")

corpus_embeddings = embedding_model.encode(
    df['combined_text'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64
).astype('float32')

print(f"Embeddings generados correctamente")
print(f"Dimension de la matriz: {corpus_embeddings.shape}")

Generando embeddings para el corpus...
Este proceso puede tomar algunos minutos


Batches: 100%|██████████| 391/391 [18:27<00:00,  2.83s/it]

Embeddings generados correctamente
Dimension de la matriz: (25000, 384)


#### Almacenamiento de embeddings

In [10]:
# Guardar embeddings y corpus procesado para uso posterior
np.save('arxiv_embeddings.npy', corpus_embeddings)
df.to_csv('arxiv_corpus_processed.csv', index=False)

print("Archivos guardados correctamente:")
print("- arxiv_embeddings.npy")
print("- arxiv_corpus_processed.csv")

Archivos guardados correctamente:
- arxiv_embeddings.npy
- arxiv_corpus_processed.csv


#### Construccion del indice FAISS

In [11]:
# Crear indice FAISS para busqueda vectorial eficiente
dimension = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(corpus_embeddings)

print(f"Indice FAISS construido con {faiss_index.ntotal} documentos")

Indice FAISS construido con 25000 documentos


#### Funcion de busqueda

In [12]:
def search_documents(query, k=10):
    """
    Busca los k documentos mas relevantes para una consulta

    Parameters:
    - query: texto de la consulta
    - k: numero de documentos a recuperar

    Returns:
    - distances: array de distancias
    - indices: array de indices de los documentos
    """
    # Limpiar la consulta
    clean_query = clean_text(query)

    # Generar embedding de la consulta
    query_vector = embedding_model.encode(
        [clean_query],
        convert_to_numpy=True
    ).astype('float32')

    # Buscar en el indice FAISS
    distances, indices = faiss_index.search(query_vector, k)

    return distances[0], indices[0]

#### Prueba de busqueda inicial

In [13]:
# Probar la funcionalidad de busqueda
test_query = "applications of deep learning in computer vision"
distances, indices = search_documents(test_query, k=5)

print(f"Consulta de prueba: {test_query}")
print("Resultados obtenidos:")
for i, idx in enumerate(indices):
    print(f"{i+1}. {df.iloc[idx]['titles'][:80]}...")
    print(f"   Terminos: {df.iloc[idx]['terms']}")
    print(f"   Distancia: {distances[i]:.4f}")
    print()

Consulta de prueba: applications of deep learning in computer vision
Resultados obtenidos:
1. ChainerCV: a Library for Deep Learning in Computer Vision...
   Terminos: ['cs.CV']
   Distancia: 0.7645

2. Deep Learning Algorithms with Applications to Video Analytics for A Smart City: ...
   Terminos: ['cs.CV']
   Distancia: 0.7957

3. Deep Learning Algorithms with Applications to Video Analytics for A Smart City: ...
   Terminos: ['cs.CV']
   Distancia: 0.7957

4. Guiding the Creation of Deep Learning-based Object Detectors...
   Terminos: ['cs.CV', 'cs.LG', 'stat.ML']
   Distancia: 0.8280

5. Deep Learning Acceleration Techniques for Real Time Mobile Vision Applications...
   Terminos: ['cs.CV', 'cs.LG', 'cs.NE']
   Distancia: 0.8292



#### Configuracion del LLM

In [15]:
import os
import google.generativeai as genai

# Leer la API key desde el archivo gapi
try:
    with open('gapi.txt', 'r') as file:
        GEMINI_API_KEY = file.read().strip()  # .strip() elimina espacios y saltos de línea
    print("API key cargada correctamente desde archivo gapi")
except FileNotFoundError:
    print("ERROR: No se encontró el archivo 'gapi'")
    GEMINI_API_KEY = None
except Exception as e:
    print(f"ERROR al leer el archivo gapi: {e}")
    GEMINI_API_KEY = None

if not GEMINI_API_KEY:
    print("ADVERTENCIA: API key no configurada")
    print("La generacion de respuestas no funcionara sin la API key")
else:
    # Configurar el cliente de Gemini
    genai.configure(api_key=GEMINI_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-3.1-flash-lite')
    print("Cliente Gemini inicializado correctamente")

API key cargada correctamente desde archivo gapi
Cliente Gemini inicializado correctamente


#### Funcion de generacion RAG

In [22]:
def generate_rag_response(query, doc_indices, k=5):
    """
    Genera una respuesta utilizando Gemini con el contexto de los documentos recuperados

    Parameters:
    - query: consulta del usuario
    - doc_indices: indices de los documentos recuperados
    - k: numero de documentos a usar como contexto

    Returns:
    - response: respuesta generada por Gemini
    """
    # Construir el contexto a partir de los documentos
    context_parts = []
    for i, idx in enumerate(doc_indices[:k]):
        title = df.iloc[idx]['titles']
        summary = df.iloc[idx]['summaries']
        terms = df.iloc[idx]['terms']

        context_parts.append(
            f"Documento {i+1}:\n"
            f"Titulo: {title}\n"
            f"Terminos: {terms}\n"
            f"Resumen: {summary[:800]}..."
        )

    context = "\n\n---\n\n".join(context_parts)

    # Sistema de prompt para garantizar fidelidad al contexto
    system_prompt = """
    Eres un asistente de investigacion especializado en articulos cientificos de arXiv.

    Instrucciones estrictas:
    1. Responde unicamente usando la informacion del contexto proporcionado.
    2. Si el contexto no contiene informacion suficiente para responder, debes decir:
       "No tengo suficiente informacion en el corpus para responder esta pregunta."
    3. No inventes datos, estadisticas o afirmaciones que no esten en el contexto.
    4. Si encuentras informacion relevante, sintetiza y organiza la respuesta.
    5. Menciona los terminos de los papers cuando sea relevante para dar contexto.
    """

    user_prompt = f"""Contexto de documentos recuperados:
{context}

Pregunta del usuario:
{query}

Respuesta (basada estrictamente en el contexto):"""

    try:
        # Crear el prompt completo para Gemini
        full_prompt = f"{system_prompt}\n\n{user_prompt}"
        response = gemini_model.generate_content(full_prompt)
        return response.text
    except Exception as e:
        return f"Error al generar respuesta: {str(e)}"

#### Prueba del sistema completo

In [23]:
# Probar el sistema con varias consultas
test_queries = [
    "What are recent advances in graph neural networks?",
    "How is reinforcement learning applied in robotics?",
    "What are the main applications of transformers in NLP?"
]

for query in test_queries:
    print("="*80)
    print(f"Consulta: {query}")
    print("="*80)

    # Recuperar documentos
    distances, indices = search_documents(query, k=10)

    # Generar respuesta
    response = generate_rag_response(query, indices, k=5)

    print(f"Respuesta generada:")
    print(response)
    print()

    print("Evidencias utilizadas (Top 3):")
    for i in range(min(3, len(indices))):
        idx = indices[i]
        print(f"{i+1}. {df.iloc[idx]['titles'][:100]}...")
        print(f"   Terminos: {df.iloc[idx]['terms']}")
        print(f"   Distancia: {distances[i]:.4f}")
    print()

Consulta: What are recent advances in graph neural networks?
Respuesta generada:
Basado en los documentos proporcionados, los avances recientes en las redes neuronales de grafos (GNN) abarcan diversas áreas de optimización, implementación y arquitectura:

**1. Optimización y entrenamiento:**
*   **Gradiente natural:** Se ha propuesto el uso de herramientas de geometría de la información para optimizar arquitecturas de GNN (específicamente *graph convolutional networks*), permitiendo explotar la geometría del espacio de parámetros para mejorar la inferencia y el aprendizaje semisupervisado (Documento 3).
*   **Normalización:** Se ha desarrollado **GraphNorm**, un método diseñado para acelerar el entrenamiento de las GNN, abordando las limitaciones de otros métodos como *InstanceNorm* (que puede degradar la expresividad en grafos regulares) o *BatchNorm* (afectado por el ruido en conjuntos de datos de grafos) (Documento 4).
*   **Convergencia:** Existe una propuesta de un algoritmo efici

#### Prueba de reconocimiento de falta de informacion

In [24]:
# Caso especial: consulta fuera del dominio del corpus
query_out_of_domain = "What are the best techniques for cooking Italian pasta?"
print("="*80)
print("Caso especial: Consulta fuera del dominio del corpus")
print("="*80)

distances, indices = search_documents(query_out_of_domain, k=5)
response = generate_rag_response(query_out_of_domain, indices, k=5)

print(f"Consulta: {query_out_of_domain}")
print(f"Respuesta del sistema:")
print(response)
print()

# Verificar si el sistema reconoce la falta de informacion
if "no tengo suficiente informacion" in response.lower():
    print("El sistema reconocio correctamente la falta de informacion")
else:
    print("El sistema no reconocio la falta de informacion")

Caso especial: Consulta fuera del dominio del corpus
Consulta: What are the best techniques for cooking Italian pasta?
Respuesta del sistema:
No tengo suficiente información en el corpus para responder esta pregunta. Los documentos proporcionados se centran en el uso de modelos generativos (como GANs), arquitecturas de aprendizaje profundo (como Inception) para el reconocimiento de estados de ingredientes de cocina, y modelos bayesianos no paramétricos, pero no contienen información sobre técnicas culinarias para preparar pasta italiana.

El sistema no reconocio la falta de informacion


#### Evaluacion cualitativa del sistema

In [31]:
# Consultas para evaluacion
evaluation_queries = [
    "How are transformers used in natural language processing?",
    "Applications of machine learning in healthcare",
    "What is the relationship between quantum computing and cryptography?",
    "Recent advances in diffusion models for image generation",
    "How is deep learning applied to drug discovery?"
]

print("""
EVALUACION CUALITATIVA DEL SISTEMA RAG

Metodologia de evaluacion:
Se evaluaran 5 consultas de prueba analizando los siguientes criterios:
1. Correccion factual de la respuesta
2. Relevancia con respecto a la consulta
3. Fidelidad al contexto recuperado
4. Capacidad de sintesis de multiples fuentes
5. Reconocimiento de falta de informacion
""")

evaluation_results = []

for i, query in enumerate(evaluation_queries, 1):
    print(f"Consulta {i}: {query}")
    print("-" * 50)

    # Recuperar documentos
    distances, indices = search_documents(query, k=5)

    # Generar respuesta
    response = generate_rag_response(query, indices, k=5)

    print(f"Respuesta generada:")
    print(response)
    print()

    print("Documentos utilizados como evidencia:")
    for j, idx in enumerate(indices[:3]):
        print(f"{j+1}. {df.iloc[idx]['titles'][:80]}...")

    print()
    print("Analisis cualitativo:")

    # Analisis especifico para cada consulta
    if i == 1:
        print("- Correccion: Correcta. La respuesta menciona usos reales de Transformers en NLP (popularidad, limitaciones estructurales, adaptaciones jerarquicas como U-Net, manejo de entradas largas con ETC, y aplicacion a codigo fuente). Todo respaldado por los resumenes recuperados.")
        print("- Relevancia: Alta. Responde directamente a la pregunta, enumerando aplicaciones y variantes.")
        print("- Fidelidad: Excelente. Cada afirmacion se apoya en documentos citados (Documentos 1-4). No hay inventos.")
        print("- Integracion: Si, sintetiza informacion de al menos 4 documentos distintos, organizandolos por tematica.")
        print("- Insuficiencia: No aplica (habia informacion suficiente).")
    elif i == 2:
        print("- Correccion: Correcta. Describe prediccion de reingresos, mortalidad, costes, clasificacion de diagnosticos, procesamiento de EHR y aprendizaje por refuerzo en salud inteligente. Todos los puntos estan en los resumenes.")
        print("- Relevancia: Muy relevante, cubre diversas areas de ML en salud.")
        print("- Fidelidad: Estricta. Se citan documentos especificos para cada aplicacion (Documento 1,2,3,4,5).")
        print("- Integracion: Combina informacion de 5 documentos diferentes.")
        print("- Insuficiencia: No aplica.")
    elif i == 3:
        print("- Correccion: Correcta (el sistema dice que no tiene informacion). Efectivamente, los documentos recuperados tratan sobre matching cuantico, redes neuronales cuanticas y problemas de correspondencia, pero no abordan criptografia.")
        print("- Relevancia: Responde adecuadamente al indicar que no hay datos suficientes.")
        print("- Fidelidad: Total, no inventa nada.")
        print("- Integracion: No aplica (no hay informacion que integrar).")
        print("- Insuficiencia: Sobresaliente. El sistema identifica claramente la falta de informacion y lo expresa con el mensaje estipulado.")
    elif i == 4:
        print("- Correccion: Correcta. Detalla avances como DDIMs (muestreo acelerado), ILVR (control de generacion), modelos en cascada (alta fidelidad), fundamentos variacionales y manejo de divergencia del score (UDM). Todo aparece en los resumenes.")
        print("- Relevancia: Total, enfocada en los avances recientes.")
        print("- Fidelidad: Basada exclusivamente en el contexto. Se citan los documentos (ILVR, Cascaded, DDIM, etc.).")
        print("- Integracion: Integra informacion de al menos 4 documentos diferentes, organizando por areas (optimizacion, control, escalabilidad, teoria).")
        print("- Insuficiencia: No aplica.")
    elif i == 5:
        print("- Correccion: Correcta. Menciona prediccion de afinidad farmaco-objetivo (DeepGS), aprendizaje de representaciones moleculares (MPG, MolGNet), uso de informacion 4D (conformadores) y mejora de interpretabilidad con indice de Gini. Todo esta en los resumenes.")
        print("- Relevancia: Responde directamente.")
        print("- Fidelidad: Estricta, con referencias a los documentos (DeepGS, MPG, etc.).")
        print("- Integracion: Sintetiza multiples enfoques de varios documentos.")
        print("- Insuficiencia: No aplica.")

    print("="*80)
    print()

    evaluation_results.append({
        'query': query,
        'response': response,
        'documents': [df.iloc[idx]['titles'] for idx in indices[:3]]
    })


EVALUACION CUALITATIVA DEL SISTEMA RAG

Metodologia de evaluacion:
Se evaluaran 5 consultas de prueba analizando los siguientes criterios:
1. Correccion factual de la respuesta
2. Relevancia con respecto a la consulta
3. Fidelidad al contexto recuperado
4. Capacidad de sintesis de multiples fuentes
5. Reconocimiento de falta de informacion

Consulta 1: How are transformers used in natural language processing?
--------------------------------------------------
Respuesta generada:
Basado en el contexto proporcionado, los Transformers se utilizan en el procesamiento del lenguaje natural (NLP) de las siguientes maneras:

*   **Rendimiento general:** Se han vuelto cada vez más populares debido a su impresionante desempeño en diversas tareas de NLP.
*   **Desafíos en la estructura:** Se argumenta que las arquitecturas Transformer estándar operan a nivel de representaciones de palabras y no intentan aprender explícitamente la estructura jerárquica, la cual se considera integral para el lengu

#### Analisis de resultados

In [30]:
print("""
ANALISIS DE RESULTADOS DEL SISTEMA RAG

Fortalezas del sistema:
1. Busqueda semantica efectiva utilizando embeddings de alta calidad
2. Recuperacion rapida gracias a FAISS
3. Generacion de respuestas contextualizadas usando Gemini
4. Sistema de prompt bien estructurado que reduce alucinaciones

Limitaciones identificadas:
1. El corpus esta limitado a aproximadamente 39,000 documentos
2. La calidad de la respuesta depende criticamente de la calidad de la recuperacion
3. El uso de API de Gemini tiene costos asociados
4. No hay memoria conversacional

Mejoras potenciales:
1. Implementar un sistema de cache para consultas frecuentes
2. Agregar filtrado por terminos para mejorar la precision
3. Implementar un sistema de feedback del usuario
4. Expandir el corpus con mas documentos

Conclusion:
El sistema RAG implementado demuestra ser efectivo para responder consultas
sobre temas cubiertos por el corpus de arXiv. La combinacion de busqueda
vectorial con generacion por LLM permite obtener respuestas contextualizadas
y relevantes. El sistema reconoce adecuadamente cuando no hay informacion
suficiente para responder una consulta.
""")


ANALISIS DE RESULTADOS DEL SISTEMA RAG

Fortalezas del sistema:
1. Busqueda semantica efectiva utilizando embeddings de alta calidad
2. Recuperacion rapida gracias a FAISS
3. Generacion de respuestas contextualizadas usando Gemini
4. Sistema de prompt bien estructurado que reduce alucinaciones

Limitaciones identificadas:
1. El corpus esta limitado a aproximadamente 39,000 documentos
2. La calidad de la respuesta depende criticamente de la calidad de la recuperacion
3. El uso de API de Gemini tiene costos asociados
4. No hay memoria conversacional

Mejoras potenciales:
1. Implementar un sistema de cache para consultas frecuentes
2. Agregar filtrado por terminos para mejorar la precision
3. Implementar un sistema de feedback del usuario
4. Expandir el corpus con mas documentos

Conclusion:
El sistema RAG implementado demuestra ser efectivo para responder consultas
sobre temas cubiertos por el corpus de arXiv. La combinacion de busqueda
vectorial con generacion por LLM permite obtener r

#### **Aplicación desplegada**

El sistema RAG está disponible en: [https://kevin-alvear-examen-rag.streamlit.app/](https://kevin-alvear-examen-rag.streamlit.app/)